手写Softmax

In [2]:
import torch

def softmax(x, dim=-1):
    # 1.为了数值稳定，先减去最大值
    x_max = torch.max(x, dim=dim, keepdim=True).values

    # 2.指数化
    exp_x = torch.exp(x - x_max)

    # 3.归一化
    return exp_x / torch.sum(exp_x, dim=dim, keepdim=True)


手写Scale Dot-Product Attention

In [3]:
import torch
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: [B, T_q, d_k]
    K: [B, T_k, d_k]
    V: [B, T_q, T_k]
    mask: [B, T_q, T_k] 或可广播到这个形状
    """

    d_k = K.size(-1)

    # 1. 计算attention score
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    # scores: [B, T_q, T_k]

    # 2. 加mask
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # 3. softmax得到attention weights
    attn_weights = torch.softmax(scores, dim=-1)

    # 4. 用attention weights加权V
    output = torch.matmul(attn_weights, V)
    # output: [B, T_q, d_v]

    return output, attn_weights


加causal mask

In [4]:
def causal_mask(T):
    return torch.tril(torch.ones(T, T)).unsqueeze(0)

B, T, d = 2, 4, 8
Q = torch.randn(B, T, d)
K = torch.randn(B, T, d)
V = torch.randn(B, T, d)

mask = causal_mask(T)
output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
print("Output shape:", output.shape)
print("attn weights shape:", attn_weights.shape)
print(attn_weights[0])

Output shape: torch.Size([2, 4, 8])
attn weights shape: torch.Size([2, 4, 4])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.0264, 0.9736, 0.0000, 0.0000],
        [0.0066, 0.7584, 0.2349, 0.0000],
        [0.6362, 0.1297, 0.1534, 0.0807]])
